In [224]:
import transformers, peft, torch, numpy as np, librosa, pathlib, re, warnings, json, gc, time
from pprint import pprint
from tqdm import tqdm
from datasets import Dataset

In [225]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [226]:
!pip install bitsandbytes

In [227]:
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

In [228]:
model_name = "Qwen/Qwen3-0.6B"
torch_dtype = torch.float16

In [229]:
sys_prompt_short = """<|im_start|>system Ты помощник лектора. Прочитай конспект и составь краткий план лекции
    в виде 3–5 пунктов. Не пиши пояснений и не используй формулы. Отвечай только на русском.
    Игнорируй не по теме: политику, мат, вопросы студентов.
    Твой ответ должен быть без использования служебных и юникод - символов, не должен содержать LaTeX, HTML или другие специальные форматы.<|im_end|>
    <|im_start|>user
"""

In [230]:
with open("dataset_checkpoint.json", "r", encoding = "utf-8") as file:
    raw_dataset = json.load(file)

formatted_data = []

for item in raw_dataset:
    formatted_data.append({
        "text" : sys_prompt_short + item['text'],
        "gpt_plan" : item['plan'],
    })

hf_dataset = Dataset.from_list(formatted_data)

In [231]:
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
    low_cpu_mem_usage=True
)

In [232]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
  #  torch_dtype=torch_dtype,
    quantization_config=bnb_config,
    device_map="auto",
    max_length=4096,
    use_cache=False,
)

In [233]:
def format_example(item):
    input_text = item['text'] + "<|im_end|>\n<|im_start|>assistant"
    target_text = item['gpt_plan'] + tokenizer.eos_token
    full_text = input_text + target_text

    tokenized = tokenizer(
        full_text,
        truncation=True,
        padding=False,
        max_length=4096,
    )

    # прописываем, чтобы модель не училась предсказывать system_srompt + prompt
    input_len = len(tokenizer(input_text, truncation=True, max_length=4096, padding=False)['input_ids'])
    labels = [-100] * input_len + tokenized['input_ids'][input_len:]

    tokenized['labels'] = labels
    return tokenized

dataset_tokenized = hf_dataset.map(format_example, remove_columns=["text", "gpt_plan"])

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

In [234]:
lora_config = peft.LoraConfig(
    r = 16,
    lora_alpha=16,
    lora_dropout = 0.1,
    task_type=peft.TaskType.CAUSAL_LM,
)

In [235]:
print(len(dataset_tokenized['input_ids'][np.random.choice(len(dataset_tokenized))]))

3569


In [236]:
model = peft.prepare_model_for_kbit_training(model)
model = peft.get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

In [237]:
trainer = transformers.Trainer(
    model=model, train_dataset=dataset_tokenized,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        warmup_steps=250, num_train_epochs=3, learning_rate=2e-4, fp16=True,
        logging_steps=1, output_dir='outputs',  gradient_checkpointing=True,
        optim="paged_adamw_8bit",dataloader_num_workers=1,
        ),
    data_collator=transformers.DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    ),
)

In [238]:
print(f"Percentage of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters())}")
allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
print(f"GPU Memory Allocated: {allocated:.2f} GB")
print(f"GPU Memory Reserved:  {reserved:.2f} GB")

Percentage of trainable parameters: 0.006065857885615251
GPU Memory Allocated: 4.19 GB
GPU Memory Reserved:  12.62 GB


In [ ]:
trainer.train()

Step,Training Loss
1,2.123200
2,2.295000
3,2.351600
4,2.107700


In [ ]:
model.eval()
model.config.use_cache = True  # включить кэширование
model.gradient_checkpointing_disable()


In [ ]:
prompt = formatted_data[np.random.choice(len(formatted_data))]["text"] + "<|im_end|>\n<|im_start|>assistant"

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )


generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = text.split("<|model_output|>", 1)[1].strip()

In [ ]:
pprint(prompt)

In [ ]:
pprint(response)

In [ ]:
model.save_pretrained("my_lecture_planner")
tokenizer.save_pretrained("my_lecture_planner")

('my_lecture_planner/tokenizer_config.json',
 'my_lecture_planner/special_tokens_map.json',
 'my_lecture_planner/chat_template.jinja',
 'my_lecture_planner/vocab.json',
 'my_lecture_planner/merges.txt',
 'my_lecture_planner/added_tokens.json',
 'my_lecture_planner/tokenizer.json')